# Prueba: Reconocedor sobre imágenes recortadas

## Objetivo
Evaluar la precisión del pipeline completo (**detectar rostro → recortar → reconocer**) sobre imágenes
**no usadas** en el registro de embeddings.

## Estructura requerida de fotos de test
```
dataset/test_faces/
├── Gerardo_Leon_Chacon/      # 2-3 fotos NUEVAS de Gerardo (no usadas en registro)
├── Kevin_Bohorquez_Huaringa/ # 2-3 fotos NUEVAS de Kevin  (no usadas en registro)
├── Miguel_Taco_Zavala/       # 2-3 fotos NUEVAS de Miguel (no usadas en registro)
└── desconocidos/             # 3-5 fotos de personas NO registradas (ej. de Kaggle LFW)
```

### ¿Qué fotos subir?
| Categoría | Qué subir | Ejemplo |
|---|---|---|
| **Fotos nuevas de cada integrante** | Selfies o fotos frontales **diferentes** a las de `known_faces/` | `gerardo_test_frontal.jpg` |
| **Condiciones difíciles (integrantes)** | Fotos de perfil, poca luz, con gorra/lentes, a distancia | `kevin_test_perfil.jpg` |
| **Personas no registradas** | Fotos de personas que NO están en el sistema. Se recomienda [LFW de Kaggle](https://www.kaggle.com/datasets/jessicali9530/lfw-funneled-deep-funneled) | `desconocido_01.jpg` |

### Criterio de aceptación
- Accuracy global ≥ **80%**
- Si es menor, evaluar: ajustar umbral o mejorar fotos de registro

---
## 1. Configuración e imports

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Raíz del proyecto (un nivel arriba de experiments/)
RAIZ = Path.cwd()
if RAIZ.name == "experiments":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.recognizer import FaceRecognizer
from src.configuracion import (
    YOLO_WEIGHTS, EMBEDDINGS_PATH, RECOGNITION_THRESHOLD
)

print(f"Raíz del proyecto: {RAIZ}")

## 2. Inicializar detector y reconocedor

In [ ]:
from ultralytics import YOLO

# --- Detector ---
# Usamos model.predict() en vez de model.track() porque son imágenes
# estáticas (track usa ByteTrack que necesita secuencia de video).
modelo_yolo = YOLO(str(YOLO_WEIGHTS))

# --- Reconocedor ---
recognizer = FaceRecognizer(
    db_path=str(EMBEDDINGS_PATH),
    threshold=RECOGNITION_THRESHOLD
)

# --- Carpeta de test ---
TEST_DIR = RAIZ / "dataset" / "test_faces"

print(f"Modelo YOLO: {YOLO_WEIGHTS}")
print(f"Umbral de reconocimiento: {RECOGNITION_THRESHOLD}")
print(f"Personas en la base: {list(recognizer.embeddings_db.keys())}")
print(f"Embeddings por persona: {({k: len(v) for k, v in recognizer.embeddings_db.items()})}")
print(f"Carpeta de test: {TEST_DIR}")
print(f"Existe: {TEST_DIR.exists()}")

---
## 3. Ejecutar pipeline: Detectar → Recortar → Reconocer

Para cada imagen de test:
1. **Detectar** rostros con YOLO (predict, no track)
2. **Recortar** el rostro con mayor confianza (con margen 30%)
3. **Reconocer** comparando contra la base de embeddings
4. **Registrar** resultado: nombre esperado vs devuelto, distancia, si fue correcto

In [ ]:
def detectar_rostro(imagen, conf=0.5):
    """Detecta rostros en una imagen estática con YOLO predict.
    
    A diferencia de FaceDetector.detect() que usa track() para video,
    aquí usamos predict() que no requiere secuencia temporal.
    
    Returns:
        Lista de tuplas (x1, y1, x2, y2, confianza).
    """
    results = modelo_yolo.predict(imagen, conf=conf, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return []
    
    boxes = results[0].boxes.xyxy.cpu().numpy()
    scores = results[0].boxes.conf.cpu().numpy()
    
    return [(int(b[0]), int(b[1]), int(b[2]), int(b[3]), float(s))
            for b, s in zip(boxes, scores)]


def recortar_rostro(imagen, deteccion, margen=0.3):
    """Recorta el rostro del frame expandiendo el bounding box.
    
    Replica la lógica de FaceDetector.crop_face() para mantener
    consistencia con el pipeline de tiempo real.
    
    Returns:
        Recorte BGR (numpy array) o None si no es válido.
    """
    x1, y1, x2, y2 = deteccion[:4]
    h, w = imagen.shape[:2]
    
    mx = int((x2 - x1) * margen)
    my = int((y2 - y1) * margen)
    
    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)
    
    if y2 > y1 and x2 > x1:
        return imagen[y1:y2, x1:x2]
    return None


print("Funciones de detección y recorte definidas ✅")

In [ ]:
# ============================================================
# EJECUTAR PIPELINE SOBRE TODAS LAS IMÁGENES DE TEST
# ============================================================

resultados = []

if not TEST_DIR.exists():
    print(f"❌ ERROR: No existe la carpeta {TEST_DIR}")
    print("Créala y agrega las fotos de test antes de ejecutar.")
else:
    carpetas = sorted([c for c in TEST_DIR.iterdir() if c.is_dir()])
    
    if not carpetas:
        print("⚠️ No hay carpetas en test_faces/. Agrega subcarpetas con fotos.")
    
    for carpeta in carpetas:
        # El nombre esperado se determina por la carpeta:
        # - Si es "desconocidos" → esperamos que devuelva "Desconocido"
        # - Si es "Gerardo_Leon_Chacon" → esperamos que devuelva ese nombre
        if carpeta.name.lower() == "desconocidos":
            nombre_esperado = "Desconocido"
        else:
            nombre_esperado = carpeta.name
        
        fotos = sorted(
            f for f in carpeta.glob("*")
            if f.suffix.lower() in (".jpg", ".jpeg", ".png")
        )
        
        if not fotos:
            print(f"⚠️ {carpeta.name}: sin fotos")
            continue
        
        print(f"\n📂 {carpeta.name} ({len(fotos)} fotos, esperado: {nombre_esperado})")
        print("-" * 60)
        
        for foto in fotos:
            imagen = cv2.imread(str(foto))
            if imagen is None:
                resultados.append({
                    "archivo": foto.name,
                    "carpeta": carpeta.name,
                    "esperado": nombre_esperado,
                    "devuelto": "Error lectura",
                    "distancia": 0.0,
                    "conf_deteccion": 0.0,
                    "correcto": False,
                    "nota": "No se pudo leer la imagen"
                })
                print(f"  ❌ {foto.name}: no se pudo leer")
                continue
            
            # --- PASO 1: Detectar rostros ---
            detecciones = detectar_rostro(imagen)
            
            if not detecciones:
                resultados.append({
                    "archivo": foto.name,
                    "carpeta": carpeta.name,
                    "esperado": nombre_esperado,
                    "devuelto": "Sin detección",
                    "distancia": 0.0,
                    "conf_deteccion": 0.0,
                    "correcto": False,
                    "nota": "YOLO no detectó ningún rostro"
                })
                print(f"  ❌ {foto.name}: YOLO no detectó rostro")
                continue
            
            # --- PASO 2: Tomar el rostro con mayor confianza ---
            mejor = max(detecciones, key=lambda d: d[4])
            conf_det = mejor[4]
            
            # --- PASO 3: Recortar ---
            crop = recortar_rostro(imagen, mejor)
            
            if crop is None:
                resultados.append({
                    "archivo": foto.name,
                    "carpeta": carpeta.name,
                    "esperado": nombre_esperado,
                    "devuelto": "Recorte inválido",
                    "distancia": 0.0,
                    "conf_deteccion": conf_det,
                    "correcto": False,
                    "nota": "El recorte del rostro no fue válido"
                })
                print(f"  ❌ {foto.name}: recorte inválido")
                continue
            
            # --- PASO 4: Reconocer ---
            nombre_devuelto, distancia = recognizer.recognize(crop)
            
            # --- PASO 5: ¿Es correcto? ---
            correcto = (nombre_devuelto == nombre_esperado)
            
            resultados.append({
                "archivo": foto.name,
                "carpeta": carpeta.name,
                "esperado": nombre_esperado,
                "devuelto": nombre_devuelto,
                "distancia": round(distancia, 4),
                "conf_deteccion": round(conf_det, 4),
                "correcto": correcto,
                "nota": ""
            })
            
            estado = "✅" if correcto else "❌"
            print(f"  {estado} {foto.name}: esperado={nombre_esperado}, "
                  f"devuelto={nombre_devuelto}, dist={distancia:.4f}, "
                  f"conf={conf_det:.4f}")

print(f"\n{'=' * 60}")
print(f"Total imágenes procesadas: {len(resultados)}")

---
## 4. Tabla de resultados y accuracy global

In [ ]:
if resultados:
    df = pd.DataFrame(resultados)
    
    # --- Tabla completa de resultados ---
    print("=" * 80)
    print("TABLA DE RESULTADOS DETALLADA")
    print("=" * 80)
    
    # Colorear filas: verde si correcto, rojo si incorrecto
    def colorear_fila(row):
        color = "background-color: #d4edda" if row["correcto"] else "background-color: #f8d7da"
        return [color] * len(row)
    
    columnas_mostrar = ["archivo", "esperado", "devuelto", "distancia",
                        "conf_deteccion", "correcto", "nota"]
    
    display(df[columnas_mostrar].style.apply(colorear_fila, axis=1))
    
    # --- Accuracy global ---
    total = len(df)
    correctos = df["correcto"].sum()
    accuracy = correctos / total * 100
    
    print(f"\n{'=' * 50}")
    print(f"ACCURACY GLOBAL: {correctos}/{total} = {accuracy:.1f}%")
    print(f"Umbral usado: {RECOGNITION_THRESHOLD}")
    print(f"{'=' * 50}")
    
    if accuracy >= 80:
        print("\n✅ CUMPLE el criterio de aceptación (≥ 80%)")
    else:
        print("\n❌ NO CUMPLE el criterio de aceptación (< 80%)")
        print("   → Evaluar: ajustar umbral o mejorar fotos de registro")
else:
    print("⚠️ No hay resultados. Agrega fotos a dataset/test_faces/ y re-ejecuta.")

## 5. Accuracy por persona

In [ ]:
if resultados:
    df = pd.DataFrame(resultados)
    
    print("ACCURACY POR PERSONA / CATEGORÍA")
    print("=" * 50)
    
    accuracy_por_persona = []
    
    for persona in df["esperado"].unique():
        subset = df[df["esperado"] == persona]
        total_p = len(subset)
        correctos_p = int(subset["correcto"].sum())
        acc_p = correctos_p / total_p * 100
        dist_prom = subset[subset["distancia"] > 0]["distancia"].mean()
        
        accuracy_por_persona.append({
            "persona": persona,
            "total_fotos": total_p,
            "correctos": correctos_p,
            "incorrectos": total_p - correctos_p,
            "accuracy_%": f"{acc_p:.1f}%",
            "dist_promedio": f"{dist_prom:.4f}" if not np.isnan(dist_prom) else "N/A"
        })
        
        icono = "✅" if acc_p >= 80 else "❌"
        print(f"  {icono} {persona}: {correctos_p}/{total_p} = {acc_p:.1f}%  "
              f"(dist promedio: {dist_prom:.4f})")
    
    print()
    df_accuracy = pd.DataFrame(accuracy_por_persona)
    display(df_accuracy)
    
    # --- Gráfico de barras ---
    fig, ax = plt.subplots(figsize=(10, 6))
    
    personas = [a["persona"] for a in accuracy_por_persona]
    # Acortar nombres para el gráfico
    personas_corto = [p.split("_")[0] if p != "Desconocido" else p for p in personas]
    accuracies = [float(a["accuracy_%"].replace("%", "")) for a in accuracy_por_persona]
    colores = ["#27ae60" if a >= 80 else "#e74c3c" for a in accuracies]
    
    bars = ax.bar(personas_corto, accuracies, color=colores,
                  edgecolor="white", linewidth=1.5, width=0.6)
    ax.axhline(y=80, color="#e67e22", linestyle="--", linewidth=2,
               label="Criterio de aceptación (80%)")
    ax.set_ylabel("Accuracy (%)", fontsize=12)
    ax.set_title("Accuracy del Reconocedor por Persona", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 110)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 2,
                f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=12)
    
    plt.tight_layout()
    ruta_grafico = RAIZ / "experiments" / "graphs" / "test_accuracy_por_persona.png"
    plt.savefig(str(ruta_grafico), dpi=150, bbox_inches="tight")
    print(f"\nGráfico guardado en: {ruta_grafico}")
    plt.show()

## 6. Distribución de distancias: Conocidos vs Desconocidos

In [ ]:
if resultados:
    df = pd.DataFrame(resultados)
    df_con_dist = df[df["distancia"] > 0].copy()
    
    if not df_con_dist.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        
        # Separar conocidos vs desconocidos (por lo que se ESPERABA)
        conocidos = df_con_dist[df_con_dist["esperado"] != "Desconocido"]["distancia"]
        desconocidos = df_con_dist[df_con_dist["esperado"] == "Desconocido"]["distancia"]
        
        if not conocidos.empty:
            ax.hist(conocidos, bins=15, alpha=0.7, color="#27ae60",
                    label=f"Conocidos (n={len(conocidos)})", edgecolor="white")
        if not desconocidos.empty:
            ax.hist(desconocidos, bins=15, alpha=0.7, color="#e74c3c",
                    label=f"Desconocidos (n={len(desconocidos)})", edgecolor="white")
        
        ax.axvline(x=RECOGNITION_THRESHOLD, color="#e67e22", linestyle="--",
                   linewidth=2, label=f"Umbral ({RECOGNITION_THRESHOLD})")
        
        ax.set_xlabel("Distancia euclidiana", fontsize=12)
        ax.set_ylabel("Frecuencia", fontsize=12)
        ax.set_title("Distribución de Distancias: Conocidos vs Desconocidos",
                     fontsize=14, fontweight="bold")
        ax.legend(fontsize=11)
        ax.grid(axis="y", alpha=0.3)
        
        plt.tight_layout()
        ruta_hist = RAIZ / "experiments" / "graphs" / "test_distribucion_distancias.png"
        plt.savefig(str(ruta_hist), dpi=150, bbox_inches="tight")
        print(f"Gráfico guardado en: {ruta_hist}")
        plt.show()
        
        print(f"\nEstadísticas de distancia:")
        if not conocidos.empty:
            print(f"  Conocidos  → min={conocidos.min():.4f}, "
                  f"max={conocidos.max():.4f}, media={conocidos.mean():.4f}")
        if not desconocidos.empty:
            print(f"  Desconocidos → min={desconocidos.min():.4f}, "
                  f"max={desconocidos.max():.4f}, media={desconocidos.mean():.4f}")

## 7. Visualización de casos fallidos

In [ ]:
if resultados:
    df = pd.DataFrame(resultados)
    fallidos = df[~df["correcto"]]
    
    if fallidos.empty:
        print("🎉 ¡No hubo casos fallidos! Accuracy = 100%")
    else:
        print(f"CASOS FALLIDOS: {len(fallidos)} de {len(df)}")
        print("=" * 50)
        
        # Mostrar tabla de fallidos
        display(fallidos[["archivo", "esperado", "devuelto", "distancia", "nota"]])
        
        # Visualizar las imágenes fallidas
        n_fallidos = len(fallidos)
        n_cols = min(4, n_fallidos)
        n_rows = max(1, (n_fallidos + n_cols - 1) // n_cols)
        
        fig, axes = plt.subplots(n_rows, n_cols,
                                figsize=(4 * n_cols, 5 * n_rows))
        if n_fallidos == 1:
            axes = np.array([axes])
        axes = np.array(axes).flatten()
        
        for idx, (_, row) in enumerate(fallidos.iterrows()):
            if idx >= len(axes):
                break
            
            ruta_foto = TEST_DIR / row["carpeta"] / row["archivo"]
            img = cv2.imread(str(ruta_foto))
            
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[idx].imshow(img_rgb)
            
            axes[idx].set_title(
                f"Esperado: {row['esperado'].split('_')[0]}\n"
                f"Devuelto: {row['devuelto'].split('_')[0]}\n"
                f"Dist: {row['distancia']:.4f}",
                fontsize=10, color="red", fontweight="bold"
            )
            axes[idx].axis("off")
        
        # Ocultar ejes vacíos
        for idx in range(n_fallidos, len(axes)):
            axes[idx].axis("off")
        
        plt.suptitle("Casos Fallidos del Reconocedor",
                     fontsize=14, fontweight="bold", color="red")
        plt.tight_layout()
        ruta_fallidos = RAIZ / "experiments" / "graphs" / "test_casos_fallidos.png"
        plt.savefig(str(ruta_fallidos), dpi=150, bbox_inches="tight")
        print(f"\nGráfico guardado en: {ruta_fallidos}")
        plt.show()

## 8. Exportar resultados a CSV

In [ ]:
if resultados:
    df = pd.DataFrame(resultados)
    
    # Guardar resultados detallados
    ruta_csv = RAIZ / "experiments" / "resultados_test.csv"
    df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
    print(f"✅ Resultados exportados a: {ruta_csv}")
    
    # Guardar resumen por persona
    ruta_resumen = RAIZ / "experiments" / "resultados_test_por_persona.csv"
    df_accuracy = pd.DataFrame(accuracy_por_persona)
    df_accuracy.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")
    print(f"✅ Resumen por persona exportado a: {ruta_resumen}")

---
## 9. Análisis e hipótesis de fallos

### Resumen de resultados
*(Completar después de ejecutar)*

| Métrica | Valor |
|---|---|
| Accuracy global | __% |
| Total imágenes | __ |
| Correctos | __ |
| Incorrectos | __ |
| Umbral usado | 0.40 |

### Hipótesis de fallos
*(Documentar cada caso incorrecto con una hipótesis de por qué falló)*

| Imagen | Tipo de error | Hipótesis |
|---|---|---|
| `ejemplo.jpg` | Falso negativo (no reconoció a conocido) | Posible causa: poca luz reduce la calidad del embedding |
| `ejemplo2.jpg` | Falso positivo (reconoció a desconocido) | Posible causa: rasgos faciales similares a persona registrada |

### Tipos de error comunes
- **Falso negativo (FN)**: El sistema no reconoce a una persona registrada → `devuelto = Desconocido` cuando debería ser el nombre
  - Causas típicas: poca luz, perfil extremo, accesorios que ocluyen el rostro, umbral demasiado estricto
- **Falso positivo (FP)**: El sistema identifica a un desconocido como alguien registrado → `devuelto = NombrePersona` cuando debería ser Desconocido
  - Causas típicas: parecido facial, umbral demasiado permisivo
- **Confusión**: El sistema identifica a una persona registrada como OTRA persona registrada
  - Causas típicas: embeddings de ambas personas son cercanos, fotos de registro insuficientes

### Conclusión
*(Completar: ¿Se cumplió el criterio de 80%? ¿Se requieren ajustes?)*

### Acciones correctivas (si accuracy < 80%)
- [ ] Ajustar el umbral `RECOGNITION_THRESHOLD` en `configuracion.py`
- [ ] Agregar más fotos de registro para las personas con bajo accuracy
- [ ] Mejorar la calidad/variedad de las fotos de registro
- [ ] Considerar usar comparación contra todos los embeddings en vez de centroides